# 024 — Proyecto: asistente neuro-simbólico explicable

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("capstone", seed=24)
assert result["kind"] == "capstone"
assert result["evidence"]
show(result)


## Solución 1 — Taxonomía

a) **Tipo 2, Symbolic[Neuro]**: el controlador es la búsqueda (simbólica); la
red es una subrutina de evaluación.
b) **Tipo 3, Neuro; Symbolic**: pipeline — la red produce símbolos (texto
estructurado) y el razonador opera después.
c) **Tipo 1, symbolic Neuro symbolic**: símbolos entran, símbolos salen, lo
neuronal media sin componente simbólico explícito.
d) **Tipo 5, Neuro_{Symbolic}**: las reglas viven DENTRO de la red como sesgo
estructural diferenciable.
e) **Tipo 3**: retrieval puntuado (proxy neuronal) → reglas y política
simbólicas con traza.


## Solución 2 — Auditoría

a) Evidencia de entrada: `retrieval.ranking` (documentos con score) — es lo
único "perceptivo" del sistema.

b) Con explicación fiel: `agent.trace` (cada acción conserva argumentos y
observación) y `safety.decisions` (cada denegación lista `reasons` como
`tool_not_allowed`). Solo score, sin explicación: los valores de
`retrieval.ranking` — el porqué de un 0,87 no es inspeccionable, solo el número.

c) `limitations` declara que es una integración didáctica local sin
persistencia, autenticación ni SLO. Son parte del contrato porque acotan qué
conclusiones soporta la evidencia: omitirlas convertiría la demo en una
afirmación de producción no comprobada.


In [ ]:
result = run_lab("capstone", seed=24)
r = result["result"]
assert "ranking" in r["retrieval"] and "trace" in r["agent"]
assert all("reasons" in d for d in r["safety"]["decisions"])
assert r["release_gate"] == "human_review_required"
print("estructura de auditoría verificada ✔")


## Solución 3 — Capa de reglas

La clave es que la explicación cite regla, umbral y valor: cada frase es
verificable contra el JSON, que es la definición operativa de "fiel".


In [ ]:
def decidir(ranking, umbral=0.5):
    fuertes = [d for d in ranking if d["score"] >= umbral]
    if fuertes:
        top = fuertes[0]
        return ("citar:" + top["document"],
                f"R1: score {top['score']:.2f} ≥ {umbral} → evidencia_fuerte; "
                f"R2: evidencia_fuerte → citar('{top['document']}')")
    mejor = max(ranking, key=lambda d: d["score"])
    return ("sin_evidencia",
            f"R3: ningún score alcanzó {umbral} (máximo {mejor['score']:.2f} "
            f"en '{mejor['document']}') → no hay evidencia suficiente")

result = run_lab("capstone", seed=24)
decision, explicacion = decidir(result["result"]["retrieval"]["ranking"])
print(decision)
print(explicacion)
assert decision.startswith("citar:")


## Solución 4 — Gate auditable

a) Mínimo: (1) **identidad y fecha** del revisor; (2) **decisión y alcance**
(aprobado/rechazado, para qué uso); (3) **evidencia consultada** (qué partes de
la traza revisó y qué criterio aplicó). Sin los tres, el gate es un sello, no
un control.

b) Fallo invisible para la traza: el retrieval puntúa alto un documento
**irrelevante pero léxicamente parecido** (bag-of-words confunde vocabulario
con relevancia). Las reglas razonan impecablemente sobre esa premisa falsa y
la traza es formalmente perfecta. El revisor humano lo atrapa leyendo el
documento citado y comparándolo con la consulta — juicio de contenido que
ninguna regla del sistema representa. Por eso la explicación fiel *habilita*
la revisión (le dice dónde mirar) pero no la reemplaza.


## Reflexión

1. Recorre el JSON del capstone y clasifica cada decisión en 'explicación fiel' o 'evidencia puntuada sin explicación'. ¿Dónde está exactamente la frontera neuro→simbólica y por qué esa interfaz acota la calidad de todo el sistema?
2. Si sustituyeras el retrieval bag-of-words por embeddings neuronales, ¿qué parte de la explicabilidad se pierde, cuál se conserva y qué habría que añadir para poder confiar en los umbrales de las reglas?
3. El gate final exige revisión humana a pesar de toda la trazabilidad. Defiende (con un ejemplo concreto de fallo tipo 'símbolo mal extraído') por qué la explicación fiel hace posible la revisión pero no la reemplaza.
